# B3 Gate15 X17 Kaggle Prompt5 Validation

Read-only validation notebook for the candidate generated by Prompt4-Kaggle.

Safety scope:
- Prompt5A: ZIP / config / manifest / sha256 lightweight validation.
- Prompt5B: full `adapter_model.safetensors` tensor validation.
- No adapter generation.
- No ZIP regeneration.
- No copy to `/kaggle/working/submission.zip`.
- No SFT/training.
- No Kaggle Submit.


In [ ]:
from pathlib import Path

EXPERIMENT_NAME = "B3_GATE15_X17_ASYMMETRIC_INPROJ_SPLIT"
GATE_COMPONENT_RANK_EXPECTED = 15
X_COMPONENT_RANK_EXPECTED = 17
IN_PROJ_TOTAL_RANK_EXPECTED = 32

BASE_DIR = Path("/kaggle/working/experiments/b3_gate15_x17")
ADAPTER_DIR = BASE_DIR / "adapter"
DIAG_DIR = BASE_DIR / "diagnostics"
ADAPTER_MODEL = ADAPTER_DIR / "adapter_model.safetensors"
ADAPTER_CONFIG = ADAPTER_DIR / "adapter_config.json"
SUBMISSION_ZIP = BASE_DIR / "submission.zip"

GOLDEN_MANIFEST = DIAG_DIR / "golden_artifact_manifest.json"
CANDIDATE_MANIFEST = DIAG_DIR / "candidate_artifact_manifest.json"
VALIDATION_PRECHECK_JSON = DIAG_DIR / "validation_precheck.json"
VALIDATION_PRECHECK_REPORT = DIAG_DIR / "validation_precheck_report.md"

print("Prompt5 validation target:", BASE_DIR)
print("adapter:", ADAPTER_MODEL)
print("config:", ADAPTER_CONFIG)
print("zip:", SUBMISSION_ZIP)


In [ ]:
import hashlib
import json
import math
import os
import re
import time
import traceback
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, UTC
from pathlib import Path

import torch
from safetensors import safe_open


def now_utc() -> str:
    return datetime.now(UTC).isoformat().replace("+00:00", "Z")


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_zip_member(zip_path: Path, member_name: str) -> str:
    h = hashlib.sha256()
    with zipfile.ZipFile(zip_path, "r") as zf:
        with zf.open(member_name, "r") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
    return h.hexdigest()


def artifact_info(path: Path) -> dict:
    return {"path": str(path), "size_bytes": path.stat().st_size, "sha256": sha256_file(path)}


def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def append_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(text)


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def list_zip_entries(path: Path) -> list[dict]:
    with zipfile.ZipFile(path, "r") as zf:
        return [
            {
                "name": info.filename,
                "file_size": info.file_size,
                "compress_size": info.compress_size,
                "crc": info.CRC,
            }
            for info in zf.infolist()
        ]


def root_pollution_check() -> dict:
    paths = [
        Path("/kaggle/working/submission.zip"),
        Path("/kaggle/working/adapter_model.safetensors"),
        Path("/kaggle/working/adapter_config.json"),
    ]
    existing = [str(p) for p in paths if p.exists()]
    return {"ok": len(existing) == 0, "existing_direct_working_artifacts": existing}


def make_stop_report(name: str, reason: str, details: dict) -> None:
    DIAG_DIR.mkdir(parents=True, exist_ok=True)
    payload = {
        "status": "STOPPED",
        "reason": reason,
        "created_at": now_utc(),
        "experiment_name": EXPERIMENT_NAME,
        "details": details,
        "training_not_performed": True,
        "kaggle_submit_not_performed": True,
        "copy_to_root_not_performed": True,
    }
    write_json(DIAG_DIR / f"{name}.json", payload)
    report = "\n".join([
        f"# {name}",
        "",
        f"- status: STOPPED",
        f"- reason: {reason}",
        f"- created_at: {payload['created_at']}",
        "- training_not_performed: true",
        "- kaggle_submit_not_performed: true",
        "- copy_to_root_not_performed: true",
        "",
        "## Details",
        "",
        "```json",
        json.dumps(details, ensure_ascii=False, indent=2),
        "```",
        "",
    ])
    (DIAG_DIR / f"{name}.md").write_text(report, encoding="utf-8")

print("Prompt5 utilities loaded.")


In [ ]:
DIAG_DIR.mkdir(parents=True, exist_ok=True)
failures = []

candidate_paths = {
    "adapter_model": ADAPTER_MODEL,
    "adapter_config": ADAPTER_CONFIG,
    "submission_zip": SUBMISSION_ZIP,
    "candidate_manifest": CANDIDATE_MANIFEST,
}
for label, path in candidate_paths.items():
    if not path.exists():
        failures.append(f"missing {label}: {path}")

if failures:
    stop_details = {
        "failures": failures,
        "expected_base_dir": str(BASE_DIR),
        "expected_files": {label: str(path) for label, path in candidate_paths.items()},
        "likely_cause": (
            "Prompt5 was run in a Kaggle session where Prompt4 candidate artifacts are not present. "
            "Kaggle /kaggle/working is session-scoped, so artifacts generated in a previous run are not "
            "available unless the same session is used or the generated experiments directory is attached/copied back."
        ),
        "recovery_steps": [
            "Run experiments/b3_gate15_x17/b3_gate15_x17_kaggle_prompt4.ipynb first in this same Kaggle session.",
            "Confirm /kaggle/working/experiments/b3_gate15_x17/adapter/adapter_model.safetensors exists.",
            "Confirm /kaggle/working/experiments/b3_gate15_x17/adapter/adapter_config.json exists.",
            "Confirm /kaggle/working/experiments/b3_gate15_x17/submission.zip exists.",
            "Confirm /kaggle/working/experiments/b3_gate15_x17/diagnostics/candidate_artifact_manifest.json exists.",
            "If Prompt4 was run in another Kaggle session, attach/copy that generated experiments directory before running Prompt5, or rerun Prompt4.",
        ],
        "forbidden_recovery_actions": [
            "Do not copy to /kaggle/working/submission.zip yet.",
            "Do not Kaggle Submit yet.",
            "Do not regenerate ZIP inside Prompt5.",
        ],
    }
    make_stop_report("STOP_REPORT_PROMPT5A", "candidate_artifact_missing", stop_details)
    print("STOP Prompt5A: candidate artifacts are missing.")
    print(json.dumps(stop_details, ensure_ascii=False, indent=2))
    raise RuntimeError(f"STOP Prompt5A: {failures}")

candidate_manifest = load_json(CANDIDATE_MANIFEST)
artifact_actual = {
    "adapter_model": artifact_info(ADAPTER_MODEL),
    "adapter_config": artifact_info(ADAPTER_CONFIG),
    "submission_zip": artifact_info(SUBMISSION_ZIP),
}

manifest_matches = {}
for key in ["adapter_model", "adapter_config", "submission_zip"]:
    expected = candidate_manifest.get(key, {})
    actual = artifact_actual[key]
    manifest_matches[key] = {
        "size_match": expected.get("size_bytes") == actual["size_bytes"],
        "sha256_match": expected.get("sha256") == actual["sha256"],
        "expected_size": expected.get("size_bytes"),
        "actual_size": actual["size_bytes"],
        "expected_sha256": expected.get("sha256"),
        "actual_sha256": actual["sha256"],
    }
    if not manifest_matches[key]["size_match"] or not manifest_matches[key]["sha256_match"]:
        failures.append(f"manifest mismatch for {key}")

zip_entries = list_zip_entries(SUBMISSION_ZIP)
entry_names = sorted(row["name"] for row in zip_entries)
expected_entries = ["adapter_config.json", "adapter_model.safetensors"]
if entry_names != expected_entries:
    failures.append(f"unexpected ZIP entries: {entry_names}")

forbidden_zip_fragments = ["README.md", "checkpoint", "diagnostics", "logs", "notebook", "csv", "jsonl", "__pycache__", ".git"]
for row in zip_entries:
    if any(fragment in row["name"] for fragment in forbidden_zip_fragments):
        failures.append(f"forbidden ZIP entry: {row['name']}")

with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
    zipped_config_bytes = zf.read("adapter_config.json")
local_config_bytes = ADAPTER_CONFIG.read_bytes()
zip_config_matches_local = zipped_config_bytes == local_config_bytes
if not zip_config_matches_local:
    failures.append("ZIP adapter_config.json differs from adapter dir adapter_config.json")

zip_model_size_matches_local = None
zip_model_sha256_matches_local = None
zip_model_sha256 = None
for row in zip_entries:
    if row["name"] == "adapter_model.safetensors":
        zip_model_size_matches_local = row["file_size"] == ADAPTER_MODEL.stat().st_size
        if not zip_model_size_matches_local:
            failures.append("ZIP adapter_model.safetensors size differs from adapter dir file")
        zip_model_sha256 = sha256_zip_member(SUBMISSION_ZIP, "adapter_model.safetensors")
        zip_model_sha256_matches_local = zip_model_sha256 == artifact_actual["adapter_model"]["sha256"]
        if not zip_model_sha256_matches_local:
            failures.append("ZIP adapter_model.safetensors sha256 differs from adapter dir file")

adapter_config = load_json(ADAPTER_CONFIG)
target_modules = adapter_config.get("target_modules", [])
required_target_modules = ["k_proj", "o_proj", "in_proj", "q_proj", "up_proj", "v_proj", "down_proj", "out_proj", "lm_head"]
config_checks = {
    "peft_type_is_lora": adapter_config.get("peft_type") == "LORA",
    "r_is_32": adapter_config.get("r") == 32,
    "lora_alpha_is_32": adapter_config.get("lora_alpha") == 32,
    "lora_dropout_is_0": adapter_config.get("lora_dropout") in (0, 0.0),
    "inference_mode_is_true": adapter_config.get("inference_mode") is True,
    "target_modules_contains_required": all(m in target_modules for m in required_target_modules),
    "target_modules_excludes_gate_x": "gate_proj" not in target_modules and "x_proj" not in target_modules,
    "target_modules": target_modules,
}
for key, ok in config_checks.items():
    if key != "target_modules" and not ok:
        failures.append(f"adapter_config check failed: {key}")

root_check = root_pollution_check()
if not root_check["ok"]:
    failures.append("root pollution detected")

golden_no_touch_check = {
    "golden_manifest_exists": GOLDEN_MANIFEST.exists(),
    "golden_manifest_recheck_note": "Golden input files are read-only Kaggle inputs; byte recheck only possible where those exact files are mounted.",
}
if GOLDEN_MANIFEST.exists():
    golden_no_touch_check["golden_manifest_loaded"] = True
else:
    golden_no_touch_check["golden_manifest_loaded"] = False

validation_precheck = {
    "status": "PASS" if not failures else "FAIL",
    "checked_at": now_utc(),
    "candidate_paths": {k: str(v) for k, v in candidate_paths.items()},
    "artifact_actual": artifact_actual,
    "manifest_matches": manifest_matches,
    "zip_entries": zip_entries,
    "zip_config_matches_local": zip_config_matches_local,
    "zip_model_size_matches_local": zip_model_size_matches_local,
    "zip_model_sha256": zip_model_sha256,
    "zip_model_sha256_matches_local": zip_model_sha256_matches_local,
    "adapter_config_checks": config_checks,
    "root_pollution_check": root_check,
    "golden_no_touch_check": golden_no_touch_check,
    "training_not_performed": True,
    "kaggle_submit_not_performed": True,
    "copy_to_root_not_performed": True,
    "failures": failures,
}
write_json(VALIDATION_PRECHECK_JSON, validation_precheck)

report = f"""# Prompt5A validation precheck report

- status: {validation_precheck['status']}
- checked_at: {validation_precheck['checked_at']}
- adapter_model: `{artifact_actual['adapter_model']['path']}` / {artifact_actual['adapter_model']['size_bytes']} bytes / `{artifact_actual['adapter_model']['sha256']}`
- adapter_config: `{artifact_actual['adapter_config']['path']}` / {artifact_actual['adapter_config']['size_bytes']} bytes / `{artifact_actual['adapter_config']['sha256']}`
- submission_zip: `{artifact_actual['submission_zip']['path']}` / {artifact_actual['submission_zip']['size_bytes']} bytes / `{artifact_actual['submission_zip']['sha256']}`
- zip_entries: {entry_names}
- zip_config_matches_local: {zip_config_matches_local}
- zip_model_size_matches_local: {zip_model_size_matches_local}
- zip_model_sha256_matches_local: {zip_model_sha256_matches_local}
- root_pollution_ok: {root_check['ok']}
- training_not_performed: true
- kaggle_submit_not_performed: true
- copy_to_root_not_performed: true

## adapter_config checks

```json
{json.dumps(config_checks, ensure_ascii=False, indent=2)}
```

## failures

```json
{json.dumps(failures, ensure_ascii=False, indent=2)}
```
"""
VALIDATION_PRECHECK_REPORT.write_text(report, encoding="utf-8")

append_text(DIAG_DIR / "run_commands.md", f"""
# Prompt5A lightweight validation

- checked_at: {validation_precheck['checked_at']}
- adapter_model: {ADAPTER_MODEL}
- adapter_config: {ADAPTER_CONFIG}
- submission_zip: {SUBMISSION_ZIP}
- status: {validation_precheck['status']}
- training_not_performed: true
- kaggle_submit_not_performed: true
- copy_to_root_not_performed: true
""")

if failures:
    make_stop_report("STOP_REPORT_PROMPT5A", "prompt5a_validation_failed", validation_precheck)
    raise RuntimeError(f"STOP Prompt5A: {failures}")

print("Prompt5A PASS")
print(json.dumps(validation_precheck, ensure_ascii=False, indent=2))


In [ ]:
precheck = load_json(VALIDATION_PRECHECK_JSON)
if precheck.get("status") != "PASS":
    make_stop_report("STOP_REPORT_PROMPT5B", "prompt5a_not_passed", {"prompt5a_status": precheck.get("status")})
    raise RuntimeError("STOP Prompt5B: Prompt5A did not pass")

failures = []
limitations = []
keys = []
dtype_counts = Counter()
shape_counts = Counter()
rank_distribution = Counter()
prefix_counts = Counter()
forbidden_pattern_counts = Counter()
expected_pattern_counts = Counter()
suffix_counts = Counter()
in_proj_rank_distribution = Counter()
rank_violations = []
zero_shape_tensor_count = 0
nan_count = 0
inf_count = 0
contains_base_model_model_model = False
contains_base_model_model_backbone = False

forbidden_patterns = [".gate_proj.", ".x_proj.", "experts.w1", "experts.w2"]
expected_patterns = [".in_proj.", ".up_proj.", ".down_proj.", ".q_proj.", ".k_proj.", ".v_proj.", ".o_proj.", ".out_proj.", "lm_head"]

lora_A = {}
lora_B = {}

try:
    with safe_open(str(ADAPTER_MODEL), framework="pt", device="cpu") as f:
        keys = list(f.keys())
        for key in keys:
            shape = tuple(f.get_slice(key).get_shape())
            tensor = f.get_tensor(key)
            dtype_counts[str(tensor.dtype)] += 1
            shape_counts[str(list(shape))] += 1
            if 0 in shape:
                zero_shape_tensor_count += 1
            if torch.is_floating_point(tensor):
                nan_count += int(torch.isnan(tensor).sum().item())
                inf_count += int(torch.isinf(tensor).sum().item())
            if "base_model.model.model" in key:
                contains_base_model_model_model = True
                prefix_counts["base_model.model.model"] += 1
            if "base_model.model.backbone" in key:
                contains_base_model_model_backbone = True
                prefix_counts["base_model.model.backbone"] += 1
            for pattern in forbidden_patterns:
                if pattern in key:
                    forbidden_pattern_counts[pattern] += 1
            for pattern in expected_patterns:
                if pattern in key:
                    expected_pattern_counts[pattern] += 1
            if key.endswith(".lora_A.weight"):
                base = key[: -len(".lora_A.weight")]
                lora_A[base] = shape
                suffix = base.split(".")[-1]
                suffix_counts[suffix] += 1
                rank = int(shape[0])
                rank_distribution[str(rank)] += 1
                if rank > 32:
                    rank_violations.append({"key": key, "rank": rank})
                if suffix == "in_proj":
                    in_proj_rank_distribution[str(rank)] += 1
            elif key.endswith(".lora_B.weight"):
                base = key[: -len(".lora_B.weight")]
                lora_B[base] = shape
except Exception as exc:
    details = {"error_type": type(exc).__name__, "error": str(exc), "traceback": traceback.format_exc()}
    make_stop_report("STOP_REPORT_PROMPT5B", "safetensors_read_failed", details)
    raise

missing_pairs = []
rank_mismatches = []
all_bases = sorted(set(lora_A) | set(lora_B))
for base in all_bases:
    if base not in lora_A or base not in lora_B:
        missing_pairs.append(base)
        continue
    a_rank = int(lora_A[base][0])
    b_rank = int(lora_B[base][1]) if len(lora_B[base]) >= 2 else None
    if a_rank != b_rank:
        rank_mismatches.append({"base": base, "a_shape": list(lora_A[base]), "b_shape": list(lora_B[base])})

adapter_config = load_json(ADAPTER_CONFIG)
target_modules = adapter_config.get("target_modules", [])
emitted_modules = set(suffix_counts)
target_modules_consistency = {
    "target_modules": target_modules,
    "emitted_modules": sorted(emitted_modules),
    "missing_target_modules": sorted(set(target_modules) - emitted_modules),
    "extra_emitted_modules": sorted(emitted_modules - set(target_modules)),
    "gate_x_absent_from_target_modules": "gate_proj" not in target_modules and "x_proj" not in target_modules,
}

in_proj_tensor_count = expected_pattern_counts[".in_proj."]
in_proj_module_count = suffix_counts["in_proj"]

svd_log_path = DIAG_DIR / "svd_compression_error.jsonl"
generation_report_path = DIAG_DIR / "generation_report.md"
static_report_path = DIAG_DIR / "static_check_report.md"
candidate_manifest = load_json(CANDIDATE_MANIFEST)
gate_x_split_evidence = {
    "candidate_manifest_gate_component_rank": candidate_manifest.get("gate_component_rank"),
    "candidate_manifest_x_component_rank": candidate_manifest.get("x_component_rank"),
    "candidate_manifest_in_proj_total_rank": candidate_manifest.get("in_proj_total_rank"),
    "svd_log_exists": svd_log_path.exists(),
    "gate_rank15_log_count": 0,
    "x_rank17_log_count": 0,
    "static_report_exists": static_report_path.exists(),
    "generation_report_exists": generation_report_path.exists(),
}
if svd_log_path.exists():
    for line in svd_log_path.read_text(encoding="utf-8").splitlines():
        if "gate_component" in line and "rank32_to_15" in line:
            gate_x_split_evidence["gate_rank15_log_count"] += 1
        if "x_component" in line and "rank32_to_17" in line:
            gate_x_split_evidence["x_rank17_log_count"] += 1
else:
    limitations.append("svd_compression_error.jsonl is unavailable; gate/x split evidence relies on manifest/static reports.")
limitations.append("Final safetensors confirms merged in_proj rank 32; original gate=15/x=17 component boundary is not directly recoverable from merged tensors alone.")

zip_entries = list_zip_entries(SUBMISSION_ZIP)
zip_entry_names = sorted(row["name"] for row in zip_entries)
submission_zip_info = artifact_info(SUBMISSION_ZIP)
adapter_model_info = artifact_info(ADAPTER_MODEL)
adapter_config_info = artifact_info(ADAPTER_CONFIG)

if len(keys) < 1000:
    failures.append("tensor count is clearly too low")
if missing_pairs:
    failures.append("missing LoRA pairs found")
if rank_mismatches:
    failures.append("LoRA A/B rank mismatches found")
if rank_violations:
    failures.append("rank > 32 violations found")
if nan_count:
    failures.append("NaN values found")
if inf_count:
    failures.append("Inf values found")
if zero_shape_tensor_count:
    failures.append("zero-shape tensors found")
if contains_base_model_model_model:
    failures.append("base_model.model.model prefix remains")
if not contains_base_model_model_backbone:
    failures.append("base_model.model.backbone prefix is absent")
if dict(forbidden_pattern_counts):
    failures.append("forbidden tensor key patterns found")
if in_proj_module_count != 23:
    failures.append(f"in_proj module count is {in_proj_module_count}, expected 23")
if in_proj_tensor_count != 46:
    failures.append(f"in_proj tensor count is {in_proj_tensor_count}, expected 46")
if dict(in_proj_rank_distribution) != {"32": 23}:
    failures.append(f"in_proj rank distribution unexpected: {dict(in_proj_rank_distribution)}")
if target_modules_consistency["missing_target_modules"]:
    failures.append("target_modules missing emitted tensors")
if not target_modules_consistency["gate_x_absent_from_target_modules"]:
    failures.append("gate_proj/x_proj remain in target_modules")
if zip_entry_names != ["adapter_config.json", "adapter_model.safetensors"]:
    failures.append(f"unexpected ZIP entries: {zip_entry_names}")

adapter_consistency = {
    "status": "PASS" if not failures else "FAIL",
    "checked_at": now_utc(),
    "num_tensors": len(keys),
    "num_lora_pairs": len(all_bases),
    "max_rank_seen": max([int(k) for k in rank_distribution] or [0]),
    "rank_distribution": dict(rank_distribution),
    "rank_violations_rank_gt_32": rank_violations,
    "missing_pairs": missing_pairs,
    "rank_mismatches": rank_mismatches,
    "dtype_counts": dict(dtype_counts),
    "shape_counts_top20": dict(shape_counts.most_common(20)),
    "nan_count": nan_count,
    "inf_count": inf_count,
    "zero_shape_tensor_count": zero_shape_tensor_count,
    "prefix_counts": dict(prefix_counts),
    "contains_base_model_model_model": contains_base_model_model_model,
    "contains_base_model_model_backbone": contains_base_model_model_backbone,
    "forbidden_pattern_counts": dict(forbidden_pattern_counts),
    "expected_pattern_counts": dict(expected_pattern_counts),
    "suffix_counts": dict(suffix_counts),
    "in_proj_tensor_count": in_proj_tensor_count,
    "in_proj_module_count": in_proj_module_count,
    "in_proj_rank_distribution": dict(in_proj_rank_distribution),
    "target_modules_consistency": target_modules_consistency,
    "gate_x_split_evidence": gate_x_split_evidence,
    "limitations": limitations,
    "failures": failures,
}
write_json(DIAG_DIR / "adapter_consistency_check.json", adapter_consistency)

submission_zip_check = {
    "status": "PASS" if zip_entry_names == ["adapter_config.json", "adapter_model.safetensors"] else "FAIL",
    "path": str(SUBMISSION_ZIP),
    "size_bytes": submission_zip_info["size_bytes"],
    "sha256": submission_zip_info["sha256"],
    "entries": zip_entries,
}
write_json(DIAG_DIR / "submission_zip_check.json", submission_zip_check)

validation_passed = not failures and precheck.get("status") == "PASS" and submission_zip_check["status"] == "PASS"
diagnostic_summary = {
    "overall_status": "PASS" if validation_passed else "FAIL",
    "prompt5a_status": precheck.get("status"),
    "prompt5b_status": adapter_consistency["status"],
    "candidate_zip_path": str(SUBMISSION_ZIP),
    "candidate_zip_sha256": submission_zip_info["sha256"],
    "adapter_model_sha256": adapter_model_info["sha256"],
    "adapter_config_sha256": adapter_config_info["sha256"],
    "max_rank_seen": adapter_consistency["max_rank_seen"],
    "num_tensors": adapter_consistency["num_tensors"],
    "num_lora_pairs": adapter_consistency["num_lora_pairs"],
    "in_proj_module_count": in_proj_module_count,
    "gate_component_rank_expected": GATE_COMPONENT_RANK_EXPECTED,
    "x_component_rank_expected": X_COMPONENT_RANK_EXPECTED,
    "in_proj_total_rank_expected": IN_PROJ_TOTAL_RANK_EXPECTED,
    "validation_passed": validation_passed,
    "safe_to_copy_to_root": validation_passed,
    "safe_to_submit_after_user_approval": validation_passed,
    "training_not_performed": True,
    "kaggle_submit_not_performed": True,
    "copy_to_root_not_performed": True,
}
write_json(DIAG_DIR / "diagnostic_summary.json", diagnostic_summary)

diff_report = f"""# Prompt5B diff report

## Golden Baseline

- Protected Golden Baseline: Public LB 0.86 user-provided baseline.
- Golden source notebook was not edited.
- Golden input artifacts were not edited.

## Candidate

- Candidate adapter: `{ADAPTER_MODEL}`
- Candidate config: `{ADAPTER_CONFIG}`
- Candidate zip: `{SUBMISSION_ZIP}`

## Intended change

- Mamba gate/x split changed from Golden 16/16 to candidate 15/17.
- Merged `in_proj` total rank remains 32.

## Validation results

- Prompt5A status: {precheck.get('status')}
- Prompt5B status: {adapter_consistency['status']}
- num_tensors: {len(keys)}
- max_rank_seen: {adapter_consistency['max_rank_seen']}
- rank violations: {len(rank_violations)}
- NaN count: {nan_count}
- Inf count: {inf_count}
- zero-shape tensor count: {zero_shape_tensor_count}
- in_proj module count: {in_proj_module_count}
- in_proj rank distribution: {dict(in_proj_rank_distribution)}
- ZIP entries: {zip_entry_names}

## Unchanged items

- q/k/v/o/up/down/out/lm_head semantics were not intentionally changed in Prompt5.
- expert unfuse was not changed in Prompt5.
- prefix conversion was not changed in Prompt5.
- adapter_config semantics were not changed in Prompt5.
- ZIP was not regenerated in Prompt5.

## Limitations

{chr(10).join('- ' + item for item in limitations)}

## Adoption judgment

Prompt5 validation status: {'PASS' if validation_passed else 'FAIL'}.

## Before Kaggle Submit

Do not submit yet. Copy-to-root and Kaggle Submit require explicit user approval after reviewing this report.
"""
(DIAG_DIR / "diff_report.md").write_text(diff_report, encoding="utf-8")

rollback = f"""# Rollback plan

## If candidate LB is worse

- Do not reuse `/kaggle/working/experiments/b3_gate15_x17/submission.zip`.
- Return to the protected Golden Baseline artifacts recorded in `golden_artifact_manifest.json`.
- Re-run the Golden Baseline notebook/script or use the known-good Golden submission source.

## Delete candidate artifacts

```bash
rm -f {ADAPTER_MODEL}
rm -f {ADAPTER_CONFIG}
rm -f {SUBMISSION_ZIP}
```

## If copy-to-root was performed later

```bash
rm -f /kaggle/working/submission.zip
```

Then copy only the approved Golden/candidate zip after explicit user approval.

## Submit policy

Kaggle Submit is allowed only after explicit user approval. Prompt5 did not submit.
"""
(DIAG_DIR / "rollback.md").write_text(rollback, encoding="utf-8")

append_text(DIAG_DIR / "run_commands.md", f"""
# Prompt5B full safetensors validation

- checked_at: {adapter_consistency['checked_at']}
- adapter_model: {ADAPTER_MODEL}
- adapter_config: {ADAPTER_CONFIG}
- submission_zip: {SUBMISSION_ZIP}
- status: {adapter_consistency['status']}
- num_tensors: {len(keys)}
- num_lora_pairs: {len(all_bases)}
- max_rank_seen: {adapter_consistency['max_rank_seen']}
- training_not_performed: true
- kaggle_submit_not_performed: true
- copy_to_root_not_performed: true
""")

if failures:
    make_stop_report("STOP_REPORT_PROMPT5B", "prompt5b_validation_failed", adapter_consistency)
    raise RuntimeError(f"STOP Prompt5B: {failures}")

print("Prompt5B PASS")
print(json.dumps(diagnostic_summary, ensure_ascii=False, indent=2))


In [ ]:
summary = load_json(DIAG_DIR / "diagnostic_summary.json")
adapter_check = load_json(DIAG_DIR / "adapter_consistency_check.json")
zip_check = load_json(DIAG_DIR / "submission_zip_check.json")
print("=== Prompt5 final summary ===")
print("Prompt5A status:", summary["prompt5a_status"])
print("Prompt5B status:", summary["prompt5b_status"])
print("num_tensors:", summary["num_tensors"])
print("num_lora_pairs:", summary["num_lora_pairs"])
print("max_rank_seen:", summary["max_rank_seen"])
print("rank violations:", adapter_check["rank_violations_rank_gt_32"])
print("NaN count:", adapter_check["nan_count"])
print("Inf count:", adapter_check["inf_count"])
print("prefix contains backbone:", adapter_check["contains_base_model_model_backbone"])
print("prefix contains model.model:", adapter_check["contains_base_model_model_model"])
print("forbidden patterns:", adapter_check["forbidden_pattern_counts"])
print("in_proj module count:", adapter_check["in_proj_module_count"])
print("in_proj rank distribution:", adapter_check["in_proj_rank_distribution"])
print("gate/x evidence:", adapter_check["gate_x_split_evidence"])
print("ZIP entries:", [row["name"] for row in zip_check["entries"]])
print("safe_to_copy_to_root:", summary["safe_to_copy_to_root"])
print("Kaggle Submit: not performed")
print("Copy-to-root: not performed")
print("Next: if user approves, Prompt6 may copy candidate zip to /kaggle/working/submission.zip.")


In [ ]:
# Optional disabled copy-to-root cell.
# Do not run unless Prompt5 passes and the user explicitly approves Prompt6.
#
# import shutil
# assert load_json(DIAG_DIR / "diagnostic_summary.json")["safe_to_copy_to_root"] is True
# shutil.copy2(SUBMISSION_ZIP, Path("/kaggle/working/submission.zip"))
# print("Copied to /kaggle/working/submission.zip")
